In [2]:
%load_ext autoreload
%autoreload 2

In [14]:
import numpy as np
import pandas as pd
import time
import scipy.stats as spst
import scipy.special as spsp
from fetch_deribit_data import *
from callibration_32model import *

import sys
sys.path.insert(sys.path.index("")+1, "C:/Users/27261/Desktop/3_Courses in PHBS/3_09_AppliedStochasticProcess/Project_sv32_EMC")
import pyfeng as pf
import pyfeng.ex as pfex
from utils import *

In [ ]:
if __name__ == "__main__":
    # 放宽窗口时间，收集充足的散点
    periods = {
        "1_Pre_Shock":  ("2025-10-10 20:55:00", "2025-10-10 21:00:00"), # 5分钟
        "2_Crash":      ("2025-10-10 21:15:00", "2025-10-10 21:20:00"), # 5分钟
        "3_Recovery":   ("2025-10-10 23:05:00", "2025-10-10 23:10:00")  # 5分钟
    }
    
    for phase_name, (start_t, end_t) in periods.items():
        print(f"\n====== 正在处理阶段: {phase_name} ({start_t} 到 {end_t}) ======")
        raw_df = fetch_deribit_trades_robust(start_t, end_t)
        
        if not raw_df.empty:
            # 1. 保存未过滤的原始数据 (Raw Data)
            raw_csv_filename = f"Deribit_RAW_{phase_name}.csv"
            raw_df.to_csv(raw_csv_filename, index=False)
            print(f"[成功] 阶段 {phase_name} 的原始成交记录 ({len(raw_df)} 条) 已保存至 {raw_csv_filename}")
            
            # 2. 清洗并保存过滤后的 OTM 数据
            print(f"[处理] 正在对 {phase_name} 进行动态 Moneyness 清洗...")
            clean_otm_df = process_and_filter_otm_dynamic(raw_df)
            
            otm_csv_filename = f"Deribit_OTM_Moneyness_{phase_name}.csv"
            clean_otm_df.to_csv(otm_csv_filename, index=False)
            print(f"[成功] 得到可用 OTM 样本 {len(clean_otm_df)} 个，已保存至 {otm_csv_filename}！")
        else:
            print(f"[警告] 阶段 {phase_name} 未拉取到数据！")


====== 正在处理阶段: 1_Pre_Shock (2025-10-10 20:55:00 到 2025-10-10 21:00:00) ======
[成功] 阶段 1_Pre_Shock 的原始成交记录 (469361 条) 已保存至 Deribit_RAW_1_Pre_Shock.csv
[处理] 正在对 1_Pre_Shock 进行动态 Moneyness 清洗...
[成功] 得到可用 OTM 样本 6 个，已保存至 Deribit_OTM_Moneyness_1_Pre_Shock.csv！

====== 正在处理阶段: 2_Crash (2025-10-10 21:15:00 到 2025-10-10 21:20:00) ======
[成功] 阶段 2_Crash 的原始成交记录 (413088 条) 已保存至 Deribit_RAW_2_Crash.csv
[处理] 正在对 2_Crash 进行动态 Moneyness 清洗...
[成功] 得到可用 OTM 样本 19 个，已保存至 Deribit_OTM_Moneyness_2_Crash.csv！

====== 正在处理阶段: 3_Recovery (2025-10-10 23:05:00 到 2025-10-10 23:10:00) ======
  [网络波动] 抓取失败 (Response ended prematurely). 第 1/5 次重试中...
  [网络波动] 抓取失败 (HTTPSConnectionPool(host='history.deribit.com', port=443): Max retries exceeded with url: /api/v2/public/get_last_trades_by_currency_and_time?currency=BTC&kind=option&start_timestamp=1760137766030&end_timestamp=1760137800000&count=1000 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.

In [16]:
# 运行代码
if __name__ == "__main__":
    # 假设闪崩发生在 2024年10月11日，我们取这之前 180 天的数据
    # 你可以修改为你实际研究的日期
    k_est, t_est = fetch_dvol_and_estimate_32_parameters(end_date_str='2025-10-11', days_lookback=180)

📥 正在从 Deribit 抓取闪崩前 180 天的 BTC DVOL 数据...
✅ 3/2 模型专属离散化回归完成！(样本数: 180 天)
---------------------------------------------
统计区间: 2025-04-13 至 2025-10-10
回归方程: dV/V = (0.0571) + (-0.3481) * V_prev
---------------------------------------------
🎯 估计的长期方差 (Theta):    0.163960 (约 40.5% IV)
🎯 估计的均值回归速度 (Kappa): 127.0590
---------------------------------------------
[附] 回归 R^2: 0.0510, p-value(斜率): 2.30e-03


In [17]:
if __name__ == "__main__":
    files = {
        "1. Pre-Shock": "Deribit_OTM_Moneyness_1_Pre_Shock.csv",
        "2. Crash":     "Deribit_OTM_Moneyness_2_Crash.csv",
        "3. Recovery":  "Deribit_OTM_Moneyness_3_Recovery.csv"
    }
    for stage, file in files.items():
        run_stage_calibration_mc(stage, file)


[1. Pre-Shock] 开始极速 MC 校准...
试探: VoV=1.5000, Rho=-0.5000 => MSE: 0.4500
试探: VoV=1.5750, Rho=-0.5000 => MSE: 0.4491
试探: VoV=1.5000, Rho=-0.5250 => MSE: 0.4495
试探: VoV=1.5750, Rho=-0.5250 => MSE: 0.4485
试探: VoV=1.6125, Rho=-0.5375 => MSE: 0.4476
试探: VoV=1.6875, Rho=-0.5125 => MSE: 0.4474
试探: VoV=1.7813, Rho=-0.5062 => MSE: 0.4463
试探: VoV=1.8188, Rho=-0.5437 => MSE: 0.4446
试探: VoV=1.9406, Rho=-0.5656 => MSE: 0.4420
试探: VoV=2.1094, Rho=-0.5344 => MSE: 0.4413
试探: VoV=2.3578, Rho=-0.5328 => MSE: 0.4383
试探: VoV=2.5172, Rho=-0.5922 => MSE: 0.4336
试探: VoV=2.8852, Rho=-0.6352 => MSE: 0.4274
试探: VoV=3.3023, Rho=-0.6023 => MSE: 0.4243
试探: VoV=3.9832, Rho=-0.6207 => MSE: 0.4170
试探: VoV=4.5105, Rho=-0.7230 => MSE: 0.4106
试探: VoV=5.5869, Rho=-0.8182 => MSE: 0.4041
试探: VoV=6.6850, Rho=-0.8037 => MSE: 0.4063
试探: VoV=5.0596, Rho=-0.7158 => MSE: 0.4078
试探: VoV=7.2123, Rho=-0.9061 => MSE: 0.4115
试探: VoV=5.5978, Rho=-0.7634 => MSE: 0.4066
试探: VoV=6.6741, Rho=-0.8585 => MSE: 0.4057
试探: VoV=5.5761, Rho=-0.8